<h1>1. Carla 사용 준비</h1>

In [1]:
import carla
import time
import math

In [2]:
client = carla.Client('localhost', 2000)
world = client.load_world('/Game/Carla/Maps/Town04')

<h1>2. 차량용 blueprint 및 생성 위치 설정</h1>

In [3]:
# ego vehicle
car1_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
car1_bp.set_attribute('role_name', 'hero')
car1_bp.set_attribute('color','255,255,255')

# 주행-좌
car2_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
car2_bp.set_attribute('role_name', 'hero')

# 주행-우
car3_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
car3_bp.set_attribute('role_name', 'hero')

# 주행-전방
car4_bp = world.get_blueprint_library().find('vehicle.mercedes.coupe_2020')
car4_bp.set_attribute('role_name', 'hero')

# 사고 1
car5_bp = world.get_blueprint_library().find('vehicle.citroen.c3')
car5_bp.set_attribute('role_name', 'hero')

# 사고 2
car6_bp = world.get_blueprint_library().find('vehicle.chevrolet.impala')
car6_bp.set_attribute('role_name', 'hero')

# AMB
car7_bp = world.get_blueprint_library().find('vehicle.ford.ambulance')
car7_bp.set_attribute('role_name', 'hero')


In [27]:
spawn_point_1 = carla.Transform(carla.Location(x=175.377335, y=14.680666, z=9.333585), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # ego
spawn_point_2 = carla.Transform(carla.Location(x=168.311737, y=18.066513, z=9.648239), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # ego- 좌
spawn_point_3 = carla.Transform(carla.Location(x=165.756546, y=11.099593, z=9.763625), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # 주행 - 우
spawn_point_4 = carla.Transform(carla.Location(x=160.377335, y=14.680666, z=9.333585), carla.Rotation(pitch=0, yaw=-179.57, roll=0)) # 주행 - 전방
spawn_point_5 = carla.Transform(carla.Location(x=-54.930981, y=13.533400, z=11.055361), carla.Rotation(pitch=-0.808291, yaw=-144.808273, roll=-0.576660)) # 사고 1
spawn_point_6 = carla.Transform(carla.Location(x=-53.498806, y=10.286665, z=11.085323), carla.Rotation(pitch=-0.896940, yaw=159.454361, roll=0.357310)) # 사고 2
spawn_point_7 = carla.Transform(carla.Location(x=-48.693542, y=16.434429, z=11.161311), carla.Rotation(pitch=-1.049438, yaw=-179.041122, roll=-0.013855)) # 앰뷸런스

In [5]:
vc_rt = carla.VehicleControl(reverse=True)
vc_rf = carla.VehicleControl(reverse=False)

In [92]:
ego = world.spawn_actor(car1_bp, spawn_point_1)

time.sleep(1)
ego.apply_control(vc_rt)
time.sleep(0.5)
ego.apply_control(vc_rf)


In [93]:
left = world.spawn_actor(car2_bp, spawn_point_2)

time.sleep(1)
left.apply_control(vc_rt)
time.sleep(0.5)
left.apply_control(vc_rf)


In [94]:
right = world.spawn_actor(car3_bp, spawn_point_3)

time.sleep(1)
right.apply_control(vc_rt)
time.sleep(0.5)
right.apply_control(vc_rf)

In [95]:
front = world.spawn_actor(car4_bp, spawn_point_4)

time.sleep(1)
front.apply_control(vc_rt)
time.sleep(0.5)
front.apply_control(vc_rf)

In [96]:
obs_1 = world.spawn_actor(car5_bp, spawn_point_5)

time.sleep(1)
obs_1.apply_control(vc_rt)
time.sleep(0.5)
obs_1.apply_control(vc_rf)

In [97]:
obs_2 = world.spawn_actor(car6_bp, spawn_point_6)

time.sleep(1)
obs_2.apply_control(vc_rt)
time.sleep(0.5)
obs_2.apply_control(vc_rf)

In [98]:
amb = world.spawn_actor(car7_bp, spawn_point_7)

time.sleep(1)
amb.apply_control(vc_rt)
time.sleep(0.5)
amb.apply_control(vc_rf)

In [91]:
# actor 제거 
actor_list = [ego, left, right, front, obs_1, obs_2, amb]
for actor in actor_list:
    actor.destroy()

<h1>3. 움직임 제어 </h1>

In [99]:
vc_straight = carla.VehicleControl(throttle = 0.7, steer = 0)

vc_right = carla.VehicleControl(throttle = 0.7, steer = 0.05)

vc_brake = carla.VehicleControl(brake = 1.0, hand_brake=True)

vc_left = carla.VehicleControl(throttle = 0.7, steer = -0.095)

vc_nn = carla.VehicleControl(throttle = 0, hand_brake=False)

In [100]:
brake_light = carla.VehicleLightState.Brake
nn = carla.VehicleLightState.NONE

In [101]:
client.start_recorder("/home/user/situation_assessment.log")

'/home/user/situation_assessment.log'

In [102]:
front.apply_control(vc_straight)
right.apply_control(vc_straight)
left.apply_control(vc_straight)
ego.apply_control(vc_straight)


while True:
    x = front.get_location().x
    y = front.get_location().y
    obs_x = obs_1.get_location().x
    obs_y = obs_1.get_location().y
    dist = math.sqrt((x - obs_x)**2 + (y - obs_y)**2)
    if round(dist) <= 50: 
        front.apply_control(vc_brake)
        front.set_light_state(brake_light)
        time.sleep(0.2)
        break

ego.apply_control(vc_brake)
ego.set_light_state(brake_light)

left.apply_control(vc_brake)
left.set_light_state(brake_light)

right.apply_control(vc_brake)
right.set_light_state(brake_light)


time.sleep(2.2)

front.apply_control(vc_nn)
front.set_light_state(nn)

ego.apply_control(vc_nn)
ego.set_light_state(nn)

left.apply_control(vc_nn)
left.set_light_state(nn)

right.apply_control(vc_nn)
right.set_light_state(nn)



In [103]:
client.stop_recorder()